<a href="https://colab.research.google.com/github/SanjaraT/Sentiment-Analysis-BERT/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Install libraries

In [ ]:
!pip install torch transformers datasets scikit-learn

# load dataet

In [ ]:
from datasets import load_dataset
dataset = load_dataset("imdb")

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

# Preprocessing

In [ ]:
# tokenization
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
  return tokenizer(example['text'], padding = 'max_length', truncation = True)

dataset = dataset.map(tokenize, batched = True)

# Load Model

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2)

# Dataet & Dataloader

In [ ]:
dataset.set_format(type = 'torch', columns = ['input_ids', 'attention_mask', 'label'])

train_loader = torch.utils.data.DataLoader(dataset['train'], batch_size = 8, shuffle = True)
test_loader = torch.utils.data.DataLoader(dataset['test'], batch_size = 8, shuffle = False)

# Training pipeline

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=5e-5)
model = model.to(device)

for epoch in range(2):
    model.train()

    loop = tqdm(train_loader, leave=True)
    total_loss = 0

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} finished | Avg Loss: {total_loss/len(train_loader):.4f}")


Epoch 1: 100%|██████████| 3125/3125 [40:58<00:00,  1.27it/s, loss=0.163]


Epoch 1 finished | Avg Loss: 0.2763


Epoch 2: 100%|██████████| 3125/3125 [40:55<00:00,  1.27it/s, loss=0.197]

Epoch 2 finished | Avg Loss: 0.1623


# Evaluation

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        labels.extend(batch['label'].numpy())

print("Accuracy:", accuracy_score(labels, preds))

Accuracy: 0.91996


# prediction

In [ ]:
text = "The movie was a bit too long but at the end i felt good!"

inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1)

print("Positive" if prediction.item() == 1 else "Negative")

Positive
